In [31]:
import os

# Notice the change here: added "_community"
from langchain_community.document_loaders import PyPDFLoader 

pdf_files = [
    r"C:\Users\souravsethi\OneDrive - Nagarro\Desktop\module6\research_papers\Attention_Is_All_You_Need.pdf",
    r"C:\Users\souravsethi\OneDrive - Nagarro\Desktop\module6\research_papers\BERT_Paper.pdf",
    r"C:\Users\souravsethi\OneDrive - Nagarro\Desktop\module6\research_papers\Llama_Paper.pdf",
    r"C:\Users\souravsethi\OneDrive - Nagarro\Desktop\module6\research_papers\LoRA_Paper.pdf",
    r"C:\Users\souravsethi\OneDrive - Nagarro\Desktop\module6\research_papers\RAG_Original.pdf"
]

documents = []

for file in pdf_files:
    loader = PyPDFLoader(file) 
    documents.extend(loader.load())

print(f"Total pages loaded: {len(documents)}")

if documents:
    print("\n--- Snippet of Page 1 Content ---")
    print(documents[0].page_content[:300]) 
    print("\n--- Page 1 Metadata ---")
    print(documents[0].metadata)

Total pages loaded: 103

--- Snippet of Page 1 Content ---
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par

--- Page 1 Metadata ---
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\souravsethi\\OneDrive - Nagarro\\Desktop\\module6\\research_papers\\Attention_Is_All_You_Need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1'}


In [32]:
import re
import os

def preprocess_documents(docs):
    """Cleans text using advanced regex and extracts custom metadata."""
    for doc in docs:
        # --- 1. Clean the Main Text Content (Your Regex Logic) ---
        text = doc.page_content
        
        # Remove Page Artifacts
        text = re.sub(r"--- PAGE \d+ ---", "", text)
        
        # Remove arXiv Metadata/Timestamps
        text = re.sub(r"arXiv:\d{4}\.\d{5}v\d+\s+\[.*?\]\s+\d+\s+[a-zA-Z]+\s+\d{4}", "", text)
        
        # Fix Hyphenation across Line Breaks
        text = re.sub(r"([a-zA-Z]+)-\s*\n\s*([a-zA-Z]+)", r"\1\2", text)
        
        # Remove Inline Citations 
        text = re.sub(r"\[\d+(?:,\s*\d+)*\]", "", text)
        
        # Clean Artifact Tokens 
        text = re.sub(r"<(?:EOS|pad)>", "", text)
        
        # Normalize Whitespace and Newlines
        text = re.sub(r"\s+", " ", text)
        
        # Update the LangChain document with the stripped text
        doc.page_content = text.strip()
        
        # --- 2. Extract Associated Metadata (Assignment Requirement) ---
        filename = os.path.basename(doc.metadata.get('source', ''))
        clean_filename = filename.replace('.pdf', '').replace('_', ' ')
        
        if doc.metadata.get('page') == 0:
             # Take the first 100 characters of the cleaned text as a headline
             doc.metadata['headline'] = doc.page_content[:100] + "..."
        else:
             doc.metadata['headline'] = f"{clean_filename} - Continued"
             
        # Search for a 4-digit year starting with 20
        year_match = re.search(r'\b(20\d{2})\b', text)
        if year_match:
             doc.metadata['publication_date'] = year_match.group(1)
        else:
             doc.metadata['publication_date'] = "Unknown"
             
    return docs

# Apply the preprocessing to our loaded documents
cleaned_documents = preprocess_documents(documents)

# Verify it worked!
print(f"Successfully cleaned {len(cleaned_documents)} pages.")
print("\n--- Cleaned Text (Page 1) ---")
print(cleaned_documents[0].page_content[:400]) 

print("\n--- Updated Metadata (Page 1) ---")
print(cleaned_documents[0].metadata)

Successfully cleaned 103 pages.

--- Cleaned Text (Page 1) ---
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish Vaswani∗ Google Brain avaswani@google.com Noam Shazeer∗ Google Brain noam@google.com Niki Parmar∗ Google Research nikip@google.com Jakob Uszkoreit∗ Google Research usz@google.com Llion Jones∗ G

--- Updated Metadata (Page 1) ---
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\souravsethi\\OneDrive - Nagarro\\Desktop\\module6\\research_papers\\Attention_Is_All_You_Need.pdf', 'total_pages': 15, 'page': 0, 'page_label

In [ ]:
!pip install langchain-openai langchain-text-splitters


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [33]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
# 1. Initialize the Splitter
# chunk_size: How large each chunk will be (1000 characters is standard)
# chunk_overlap: How much of the previous chunk will be repeated in the next chunk (to avoid breaking context)
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=300,
    separators=["\n\n", "\n", "(?<=\. )", " ", ""] 
)

# 2. Apply the splitter to your 'cleaned_documents' 
# (Make sure this variable name matches your cleaning output)
chunks = text_splitter.split_documents(cleaned_documents)

# --- VERIFICATION ---
print(f"✅ Total pages after cleaning: {len(cleaned_documents)}")
print(f"✅ Total chunks after chunking: {len(chunks)}")

# Let's see if metadata (like filename/year) is still there
if chunks:
    print("\n--- Metadata check of Chunk 1 ---")
    print(chunks[0].metadata)
    
    print("\n--- Text Snippet of Chunk 1 ---")
    print(chunks[0].page_content[:300] + "...")

✅ Total pages after cleaning: 103
✅ Total chunks after chunking: 313

--- Metadata check of Chunk 1 ---
{'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2024-04-10T21:11:43+00:00', 'author': '', 'keywords': '', 'moddate': '2024-04-10T21:11:43+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'C:\\Users\\souravsethi\\OneDrive - Nagarro\\Desktop\\module6\\research_papers\\Attention_Is_All_You_Need.pdf', 'total_pages': 15, 'page': 0, 'page_label': '1', 'headline': 'Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and...', 'publication_date': '2014'}

--- Text Snippet of Chunk 1 ---
Provided proper attribution is provided, Google hereby grants permission to reproduce the tables and figures in this paper solely for use in journalistic or scholarly works. Attention Is All You Need Ashish 

In [37]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# --- STEP 4: EMBEDDING & VECTOR STORAGE ---
print("\n--- Starting Embedding Process ---")

# 1. Initialize the Embedding Model
# This model will convert text into a list of 384 numbers
embeddings_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# 2. Create Vector Database (This is where the actual embedding action happens)
# 'chunks' -> your text pieces
# 'embedding' -> your model that will generate numbers
# 'persist_directory' -> folder where data will be saved
vector_db = Chroma.from_documents(
    documents=chunks, 
    embedding=embeddings_model,
    persist_directory="./research_paper_db" 
)

print("✅ Embeddings have been generated and saved in './research_paper_db'!")

# --- STEP 5: QUICK TEST (Optional) ---
query = "What is the main idea of the Attention paper?"
search_results = vector_db.similarity_search(query, k=2)

print("\n--- Test Search Result ---")
print(f"Most relevant chunk found: {search_results[0].page_content[:200]}...")


--- Starting Embedding Process ---


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5176.42it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embeddings have been generated and saved in './research_paper_db'!

--- Test Search Result ---
Most relevant chunk found: than i. 3.2 Attention An attention function can be described as mapping a query and a set of key-value pairs to an output, where the query, keys, values, and output are all vectors. The output is comp...


In [ ]:
import os
from dotenv import load_dotenv
from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# Load environment variables from .env file
load_dotenv()

# 1. Setup LLM (Groq)
llm = ChatGroq(model_name="llama-3.3-70b-versatile", temperature=0.3)

# 2. Simple Retriever (Set k=5 for better results)
retriever = vector_db.as_retriever(search_kwargs={"k": 5})

# 3. Formatter Function (Also shows Source and Page)
def format_docs(docs):
    formatted = []
    for doc in docs:
        source = os.path.basename(doc.metadata.get('source', 'Unknown'))
        page = doc.metadata.get('page', 'Unknown')
        formatted.append(f"[Source: {source}, Page: {page}]\nContent: {doc.page_content}")
    return "\n\n".join(formatted)

# 4. Professional Prompt (Assignment Requirement: Groundedness)
template = """You are a Research Assistant. Use the following context to answer the question.
If the answer isn't in the context, just say you don't know. 
Always cite the Source and Page number in your answer.

Context:
{context}

Question: {question}

Answer:"""

rag_prompt = PromptTemplate.from_template(template)

# 5. Final Balanced RAG Chain
# This is simple but works with Metadata
final_rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | rag_prompt
    | llm
    | StrOutputParser()
 )

# --- TEST ---
print("Final RAG is ready!")
query = "What is BERT and its key features?"
response = final_rag_chain.invoke(query)
print("\n=== FINAL ANSWER ===\n", response)

Final RAG is ready!

=== FINAL ANSWER ===
 BERT is a language model with a unified architecture across different tasks. Its key features include a multi-layer bidirectional Transformer encoder, and it is trained on unlabeled data over different pre-training tasks. BERT uses a fine-tuning approach, where the model is first initialized with pre-trained parameters and then fine-tuned using labeled data from downstream tasks (Source: BERT_Paper.pdf, Page: 2). Additionally, BERT's representations are jointly conditioned on both left and right context in all layers, which is a distinctive feature of the model (Source: BERT_Paper.pdf, Page: 12).


In [47]:
# ===== SIMPLE FILE SAVE =====
print("💾 Saving RAG answers to file...")

# Same questions as previous evaluation
questions = [
    "What is BERT and how does it work?",
    "Explain the self-attention mechanism in Transformers",
    "What are the key innovations of the Transformer architecture?", 
    "How does LoRA improve fine-tuning efficiency?",
    "What is RAG and how does it combine retrieval with generation?",
    "Compare BERT and GPT model architectures",
    "How does positional encoding work in Transformers?",
    "What are the main contributions of the Attention paper?",
    "Who is Modi Ji",
    "RAG Architecture ?"
]

# Open file and write
with open("rag_answers.txt", "w", encoding="utf-8") as file:
    file.write("RAG SYSTEM ANSWERS WITH SCORES\n")
    file.write("="*50 + "\n\n")
    
    all_scores = []
    
    for i, question in enumerate(questions, 1):
        print(f"Processing question {i}...")
        
        # Get answer and docs
        answer = final_rag_chain.invoke(question)
        docs = vector_db.similarity_search(question, k=3)
        
        # Calculate scores
        context_text = " ".join(doc.page_content for doc in docs).lower()
        prompt_words = set(question.lower().split())
        answer_words = set(answer.lower().split())
        context_words = set(context_text.split())
        
        c_score = min(5, len(prompt_words & context_words) + 2)  # Context relevance
        a_score = min(5, 2 + len(answer) // 30 + bool(prompt_words & answer_words))  # Answer relevance
        g_score = min(5, len(answer_words & context_words) // 15 + 2)  # Groundedness
        
        overall = (c_score + a_score + g_score) / 3
        all_scores.append([c_score, a_score, g_score, overall])
        
        # Write to file
        file.write(f"Q{i}: {question}\n")
        file.write(f"Answer: {answer}\n")
        file.write(f"Scores - Context: {c_score}/5, Answer: {a_score}/5, Ground: {g_score}/5, Overall: {overall:.1f}/5\n")
        file.write("-"*30 + "\n\n")
    
    # Add summary
    avg_scores = [sum(col) / len(all_scores) for col in zip(*all_scores)]
    file.write("\nSUMMARY:\n")
    file.write("="*20 + "\n")
    file.write(f"Average Context Relevance: {avg_scores[0]:.1f}/5\n")
    file.write(f"Average Answer Relevance: {avg_scores[1]:.1f}/5\n")
    file.write(f"Average Groundedness: {avg_scores[2]:.1f}/5\n")
    file.write(f"Overall Average: {avg_scores[3]:.1f}/5\n")

print("✅ Done! Check 'rag_answers.txt' file")
print("📁 File saved with answers and scores")

💾 Saving RAG answers to file...
Processing question 1...
Processing question 2...
Processing question 3...
Processing question 4...
Processing question 5...
Processing question 6...
Processing question 7...
Processing question 8...
Processing question 9...
Processing question 10...
✅ Done! Check 'rag_answers.txt' file
📁 File saved with answers and scores


In [48]:
# ===== SIMPLE MULTI-QUERY RAG =====
def simple_multi_query(question):
    """Simple Multi-Query: Ask 2 more questions, get more docs, better answer"""
    
    try:
        # Generate 2 similar questions
        prompt = f"Make 2 similar questions for: {question}\nJust give 2 questions, nothing else."
        response = llm.invoke(prompt)
        text = response.content if hasattr(response, 'content') else str(response)
        
        # Use all questions to search
        all_questions = [question] + text.split('\n')[:2]
        
        # Get unique documents
        all_docs = []
        seen = set()
        for q in all_questions:
            docs = vector_db.similarity_search(q.strip(), k=2)
            for doc in docs:
                key = doc.page_content[:100]  # Use first 100 chars as key
                if key not in seen:
                    seen.add(key)
                    all_docs.append(doc)
        
        # Create context and get answer
        context = "\n\n".join([f"[{os.path.basename(doc.metadata.get('source', 'Unknown'))}]: {doc.page_content}" 
                              for doc in all_docs[:4]])
        
        answer_prompt = f"Context:\n{context}\nQuestion: {question}\nAnswer:"
        answer = llm.invoke(answer_prompt)
        
        return answer.content if hasattr(answer, 'content') else str(answer), len(all_docs)
        
    except Exception as e:
        print(f"Error: {e}")
        # Fallback to normal search
        docs = vector_db.similarity_search(question, k=3)
        return final_rag_chain.invoke(question), len(docs)

print("Simple Multi-Query RAG ready!")

Simple Multi-Query RAG ready!


In [ ]:
# ===== COMPARE NORMAL vs MULTI-QUERY RAG =====
print("🔥 COMPARING NORMAL vs MULTI-QUERY RAG")
print("="*50)

# Test questions
test_questions = [
    "What is BERT and how does it work?", 
    "How does LoRA improve fine-tuning efficiency?"
]

def calculate_score(question, answer, docs):
    """Calculate simple RAG score"""
    context_text = " ".join(doc.page_content for doc in docs).lower()
    prompt_words = set(question.lower().split())
    answer_words = set(answer.lower().split())
    context_words = set(context_text.split())
    
    c_score = min(5, len(prompt_words & context_words) + 2)  # Context relevance
    a_score = min(5, 2 + len(answer) // 30 + bool(prompt_words & answer_words))  # Answer relevance
    g_score = min(5, len(answer_words & context_words) // 15 + 2)  # Groundedness
    
    overall = (c_score + a_score + g_score) / 3
    return overall

# Compare both methods
normal_scores = []
multi_scores = []

for i, question in enumerate(test_questions, 1):
    print(f"\n Question {i}: {question}")
    print("-"*40)
    
    # Normal RAG
    normal_answer = final_rag_chain.invoke(question)
    normal_docs = vector_db.similarity_search(question, k=3)
    normal_score = calculate_score(question, normal_answer, normal_docs)
    normal_scores.append(normal_score)
    
    print(f"🔹 Normal RAG Score: {normal_score:.1f}/5 (Used {len(normal_docs)} docs)")
    
    # Multi-Query RAG  
    multi_answer, multi_doc_count = simple_multi_query(question)
    multi_docs = vector_db.similarity_search(question, k=multi_doc_count)
    multi_score = calculate_score(question, multi_answer, multi_docs)
    multi_scores.append(multi_score)
    
    print(f"🔸 Multi-Query Score: {multi_score:.1f}/5 (Used {multi_doc_count} docs)")
    
    if multi_score > normal_score:
        print("Multi-Query WINS!")
    elif normal_score > multi_score:
        print("Normal RAG WINS!")
    else:
        print("It's a TIE!")

# Final comparison
print(f"\n{'='*50}")
print("FINAL RESULTS")
print("="*50)

normal_avg = sum(normal_scores) / len(normal_scores)
multi_avg = sum(multi_scores) / len(multi_scores)

print(f"Normal RAG Average: {normal_avg:.1f}/5")
print(f"Multi-Query Average: {multi_avg:.1f}/5")

if multi_avg > normal_avg:
    print("WINNER: Multi-Query RAG performs better!")
elif normal_avg > multi_avg:
    print("WINNER: Normal RAG performs better!")
else:
    print("Both methods perform equally well!")

🔥 COMPARING NORMAL vs MULTI-QUERY RAG

 Question 1: What is BERT and how does it work?
----------------------------------------
🔹 Normal RAG Score: 5.0/5 (Used 3 docs)
🔸 Multi-Query Score: 5.0/5 (Used 3 docs)
It's a TIE!

 Question 2: How does LoRA improve fine-tuning efficiency?
----------------------------------------
🔹 Normal RAG Score: 4.3/5 (Used 3 docs)
🔸 Multi-Query Score: 4.7/5 (Used 2 docs)
Multi-Query WINS!

FINAL RESULTS
Normal RAG Average: 4.7/5
Multi-Query Average: 4.8/5
WINNER: Multi-Query RAG performs better!
